reference
- https://github.com/huggingface/transformers/blob/master/src/transformers/modeling_albert.py
- https://github.com/google-research/ALBERT/blob/master/modeling.py

In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
from functools import reduce

In [2]:
def get_num_params(model):
    n_params = 0
    for param in model.parameters():
        param_shape = list(param.shape)
        n_params += reduce(lambda x, y: x*y, param_shape)
    return n_params

In [153]:
class Config():
    def __init__(self):
        '''
            this configuration follows ALBERT `base`.
        '''
        self.n_embbed = 20 # E. originally 128
        self.n_vocab = 10  # V. originally 32000
        self.n_hidden = 40 # H. originally 768
        self.n_hidden_ff = self.n_hidden * 4 # 4H. originally n_hidden * 4
        self.n_maxseq = 15 # originally 512
        self.dropout_rate = 0.1
        self.eps = 1e-6
        self.n_layers = 6       # originally 12
        self.n_head = 8         # originally ??
        assert self.n_hidden % self.n_head == 0
        self.attention_size = self.n_hidden // self.n_head

In [4]:
class Config():
    def __init__(self):
        '''
            this configuration follows ALBERT `base`.
        '''
        self.n_embbed = 128
        self.n_vocab = 32000
        self.n_hidden = 768
        self.n_hidden_ff = self.n_hidden * 4 # 4H. originally n_hidden * 4
        self.n_maxseq = 512
        self.dropout_rate = 0.1
        self.eps = 1e-6
        self.n_layers = 12
        self.n_head = self.n_hidden//64
        assert self.n_hidden % self.n_head == 0
        self.attention_size = self.n_hidden // self.n_head

In [154]:
conf = Config()

In [22]:
tmp = torch.rand((conf.n_maxseq, conf.n_embbed))
tmp.shape, tmp[1:5].shape
tmp[1:5].shape

torch.Size([4, 128])

In [7]:
# class LayerNorm(nn.Module):
#     def __init__(self, conf, eps=1e-6):
#         super().__init__()
#         self.gamma = nn.Parameter(torch.ones(conf.n_embbed))
#         self.beta = nn.Parameter(torch.zeros(conf.n_embbed))
#         self.eps = eps
#     def forward(self, x):
#         u = x.mean(-1, keepdim=True)
#         s = (x - u).pow(2).mean(-1, keepdim=True)
#         x = (x - u) / torch.sqrt(s + self.eps)
#         return self.gamma * x + self.beta

In [95]:
class ContextDependentEmbedding(nn.Module):
    def __init__(self, conf, use_positinal_embedding=True):
        super().__init__()
        # context-independent embedding
        self.embedded_word = nn.Embedding(conf.n_vocab, conf.n_embbed)
        
        # FIXME: n_embbed -> n_embed
        self.pos_emb_vector = nn.Embedding(conf.n_maxseq, conf.n_embbed)
        self.use_positinal_embedding = use_positinal_embedding
        self.norm = nn.LayerNorm(conf.n_embbed, eps=conf.eps)
        self.drop = nn.Dropout(conf.dropout_rate)
        
    def forward(self, x):
        n_seq = x.size()[1]
        ci_emb = self.embedded_word(x)
        
        if self.use_positinal_embedding:
            cd_emb = ci_emb + self.pos_emb_vector.weight[:n_seq]
            cd_emb = self.drop(self.norm(cd_emb))
        return cd_emb

In [96]:
inp = np.random.randint(0, 10, (3))
inp = 3
inp = torch.LongTensor(range(inp))
print(inp)
ee = nn.Embedding(conf.n_maxseq, conf.n_embbed)
ee.weight[:10].shape

tensor([0, 1, 2])


torch.Size([10, 128])

In [646]:
class AlbertMultiHeadAttention(nn.Module):
    def __init__(self, conf):
        super().__init__()
        torch.manual_seed(7)
        
        self.query = nn.Linear(conf.n_hidden, conf.n_hidden)
        self.key = nn.Linear(conf.n_hidden, conf.n_hidden)
        self.value = nn.Linear(conf.n_hidden, conf.n_hidden)
        self.attention_size = conf.attention_size
        self.hidden_size = conf.n_hidden
        self.n_head = conf.n_head
        self.softmax = torch.nn.Softmax(dim=-1)
        self.layer_norm = nn.LayerNorm(conf.n_hidden, eps=conf.eps)
        
        #torch.manual_seed(7)
        #self.dense = nn.Linear(conf.n_hidden, conf.n_hidden)
        
        torch.manual_seed(7)
        self.dense = nn.Linear(conf.n_hidden, conf.n_hidden)
        #self.dense.weight = nn.Parameter(self.dense.weight)
        self.attention_head_size = conf.n_hidden // conf.n_head
        
    def split_tensor(self, x):
        x_split = torch.split(x, self.attention_size, dim=2)
        return torch.stack(x_split, dim=-2)
        
    def merge_tensor(self, x):
        s = x.size()[-2]
        return torch.cat([x[:,:,i,:] for i in range(s)], dim=-1)
    
    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        q, k, v = list(map(lambda x: self.split_tensor(x).transpose(1,2), [q, k, v]))

        score = (q @ k.transpose(-2,-1)) / np.sqrt(self.attention_size)
        prob = self.softmax(score)
        
        # FIXME: why contiguous?
        context = (prob @ v).transpose(1,2).contiguous()
        projected_context = self.dense(self.merge_tensor(context))
        print(projected_context.sum())
        
        # do something about `mask`
        hidden = self.layer_norm(x + projected_context)
        return hidden

In [647]:
# attention_head_size = conf.n_hidden // conf.n_head
# dense = nn.Linear(conf.n_hidden, conf.n_hidden) # dense = nn.Linear(A, B) -> dense.weight = (B,A) dense.bias = (B,)
# dense.weight = nn.Parameter(dense.weight.t())
# w = (
#     dense.weight.t()
#     .view(conf.n_head, attention_head_size, conf.n_hidden)
#     .to(torch.float32)
# )

In [648]:
# AlbertMultiHeadAttention test code
np.random.seed(102)
n_batch = 3
inp = np.random.random((n_batch, 10, 40))
inp = torch.Tensor(inp)

att = AlbertMultiHeadAttention(conf)
o1 = att(inp)
o1.shape, o1.sum()

tensor(46.1405, grad_fn=<SumBackward0>)


(torch.Size([3, 10, 40]), tensor(-2.3246e-06, grad_fn=<SumBackward0>))

In [591]:
np.random.seed(102)
n_batch = 3
inp = np.random.random((n_batch, 10, 40))
inp = torch.Tensor(inp)

# (3,10,40) @ (40,40)
dense(inp).shape

torch.Size([3, 10, 40])

In [489]:
def gelu_new(x):
    """ Implementation of the gelu activation function currently in Google Bert repo (identical to OpenAI GPT).
        Also see https://arxiv.org/abs/1606.08415
    """
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))

In [490]:
class AlbertLayer(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.mhead_attention = AlbertMultiHeadAttention(conf)
        self.n_hidden = conf.n_hidden
        self.n_hidden_ff = conf.n_hidden_ff
        self.feedforward = nn.Linear(conf.n_hidden, conf.n_hidden_ff)
        self.feedforward_out = nn.Linear(conf.n_hidden_ff, conf.n_hidden)
        self.act_fn = gelu_new
        
    def forward(self, x):
        x = self.mhead_attention(x)
        x = self.feedforward_out(self.act_fn(self.feedforward(x)))
        return x

In [491]:
# # AlbertLayer test code
# np.random.seed(102)
# n_batch = 3
# inp = np.random.random((n_batch, 10, 40))
# inp = torch.Tensor(inp)

# lay = AlbertLayer(conf)
# o = lay(inp)
# o.shape

In [492]:
class Transformer(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.linear_to_hidden = nn.Linear(conf.n_embbed, conf.n_hidden)
        self.n_layers = conf.n_layers
        layer = AlbertLayer(conf)
        self.layers = nn.ModuleList([layer for _ in range(self.n_layers)])
        
    def forward(self, x):
        x = self.linear_to_hidden(x)
        for layer in self.layers:
            x = layer(x)
        return x

In [493]:
class AlbertModel(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.embedding = ContextDependentEmbedding(conf, use_positinal_embedding=True)
        self.encoder = Transformer(conf)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.encoder(x)
        return x

In [494]:
np.random.seed(102)
n_batch = 3
inp = np.random.randint(0, conf.n_vocab, (n_batch, 10))
inp[:,0] = 1
inp[:,-3:] = 0
inp = torch.LongTensor(inp)
inp

tensor([[1, 3, 2, 2, 2, 9, 8, 0, 0, 0],
        [1, 7, 0, 6, 2, 7, 3, 0, 0, 0],
        [1, 7, 9, 7, 4, 9, 3, 0, 0, 0]])

In [495]:
model = AlbertModel(conf)

In [496]:
model(inp).shape

torch.float32 tensor(0.1269, grad_fn=<SumBackward0>) torch.Size([8, 5, 40])


AttributeError: 'tuple' object has no attribute 'dim'

In [497]:
get_num_params(model)

21020

# 낙서장

In [638]:
class BertSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        torch.manual_seed(7)
        
        if config.n_hidden % config.n_head != 0:
            raise ValueError(
                "The hidden size (%d) is not a multiple of the number of attention "
                "heads (%d)" % (config.n_hidden, config.n_head)
            )
        self.output_attentions = True

        self.num_attention_heads = config.n_head
        self.attention_head_size = int(config.n_hidden / config.n_head)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(config.n_hidden, self.all_head_size)
        self.key = nn.Linear(config.n_hidden, self.all_head_size)
        self.value = nn.Linear(config.n_hidden, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def transpose_for_scores(self, x):
        print(x.shape)
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        print(new_x_shape)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
    ):
        mixed_query_layer = self.query(hidden_states)

        # If this is instantiated as a cross-attention module, the keys
        # and values come from an encoder; the attention mask needs to be
        # such that the encoder's padding tokens are not attended to.
        if encoder_hidden_states is not None:
            mixed_key_layer = self.key(encoder_hidden_states)
            mixed_value_layer = self.value(encoder_hidden_states)
            attention_mask = encoder_attention_mask
        else:
            mixed_key_layer = self.key(hidden_states)
            mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # Take the dot product between "query" and "key" to get the raw attention scores.
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        if attention_mask is not None:
            # Apply the attention mask is (precomputed for all layers in BertModel forward() function)
            attention_scores = attention_scores + attention_mask

        # Normalize the attention scores to probabilities.
        attention_probs = nn.Softmax(dim=-1)(attention_scores)

        # This is actually dropping out entire tokens to attend to, which might
        # seem a bit unusual, but is taken from the original Transformer paper.
        #attention_probs = self.dropout(attention_probs)

        # Mask heads if we want to
        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)

        outputs = (context_layer, attention_probs, context_layer) if self.output_attentions else (context_layer,)
        return outputs

In [639]:
class AlbertAttention(BertSelfAttention):
    def __init__(self, config):
        super().__init__(config)
        torch.manual_seed(7)
        
        self.output_attentions = True
        self.num_attention_heads = config.n_head
        self.hidden_size = config.n_hidden
        self.attention_head_size = config.n_hidden // config.n_head
        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)
        
        torch.manual_seed(7)
        self.dense = nn.Linear(config.n_hidden, config.n_hidden)
        self.LayerNorm = nn.LayerNorm(config.n_hidden, eps=config.eps)
        self.pruned_heads = set()

    '''
    
    def prune_heads(self, heads):
        if len(heads) == 0:
            return
        mask = torch.ones(self.num_attention_heads, self.attention_head_size)
        heads = set(heads) - self.pruned_heads  # Convert to set and emove already pruned heads
        for head in heads:
            # Compute how many pruned heads are before the head and move the index accordingly
            head = head - sum(1 if h < head else 0 for h in self.pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()

        # Prune linear layers
        self.query = prune_linear_layer(self.query, index)
        self.key = prune_linear_layer(self.key, index)
        self.value = prune_linear_layer(self.value, index)
        self.dense = prune_linear_layer(self.dense, index, dim=1)

        # Update hyper params and store pruned heads
        self.num_attention_heads = self.num_attention_heads - len(heads)
        self.all_head_size = self.attention_head_size * self.num_attention_heads
        self.pruned_heads = self.pruned_heads.union(heads)
    '''
    
    def forward(self, input_ids, attention_mask=None, head_mask=None):
        mixed_query_layer = self.query(input_ids)
        mixed_key_layer = self.key(input_ids)
        mixed_value_layer = self.value(input_ids)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # Take the dot product between "query" and "key" to get the raw attention scores.
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        if attention_mask is not None:
            # Apply the attention mask is (precomputed for all layers in BertModel forward() function)
            attention_scores = attention_scores + attention_mask

        # Normalize the attention scores to probabilities.
        attention_probs = nn.Softmax(dim=-1)(attention_scores)

        # This is actually dropping out entire tokens to attend to, which might
        # seem a bit unusual, but is taken from the original Transformer paper.
        #attention_probs = self.dropout(attention_probs)

        # Mask heads if we want to
        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()

        torch.manual_seed(7)
        # Should find a better way to do this
        w = (
            self.dense.weight.t()
            .view(self.num_attention_heads, self.attention_head_size, self.hidden_size)
            .to(context_layer.dtype)
        )
        b = self.dense.bias.to(context_layer.dtype)

        projected_context_layer = torch.einsum("bfnd,ndh->bfh", context_layer, w) + b
        print(projected_context_layer.sum())
        projected_context_layer_dropout = self.dropout(projected_context_layer)
        layernormed_context_layer = self.LayerNorm(input_ids + projected_context_layer_dropout)
        return layernormed_context_layer

In [640]:
# AlbertMultiHeadAttention test code
np.random.seed(102)
n_batch = 3
inp = np.random.random((n_batch, 10, conf.n_hidden))
inp = torch.Tensor(inp)

In [641]:
conf.attention_probs_dropout_prob = 0.0
att = AlbertAttention(conf)
o = att(inp)
o.shape, o.sum()

torch.Size([3, 10, 40])
torch.Size([3, 10, 8, 5])
torch.Size([3, 10, 40])
torch.Size([3, 10, 8, 5])
torch.Size([3, 10, 40])
torch.Size([3, 10, 8, 5])
tensor(46.1405, grad_fn=<SumBackward0>)


(torch.Size([3, 10, 40]), tensor(-2.3246e-06, grad_fn=<SumBackward0>))

In [642]:
# AlbertMultiHeadAttention test code
np.random.seed(102)
n_batch = 3
inp = np.random.random((n_batch, 10, 40))
inp = torch.Tensor(inp)

att = AlbertMultiHeadAttention(conf)
o1 = att(inp)
o1.shape, o1.sum()

(torch.Size([3, 10, 40]), tensor(-2.3246e-06, grad_fn=<SumBackward0>))

In [565]:
o[:,0,:20]

tensor([[ 1.1431,  0.9443, -1.1135,  0.9790,  0.8699,  0.7742, -1.2484,  0.3398,
         -1.3116,  0.8435, -0.5928,  2.4487, -1.5294, -0.1209,  0.8277,  0.2153,
          0.4295, -0.5154, -0.3021, -1.0291],
        [ 1.9075, -0.5780, -0.1342,  0.4117,  1.6919, -0.7286, -1.1590,  0.3912,
         -0.8881, -0.0081,  0.2495,  0.3791,  0.6632, -0.2955, -0.8054, -0.4049,
          0.2013, -0.4917,  0.0273, -0.7945],
        [ 1.4742,  0.3332, -0.9849, -0.1476,  0.7846, -0.7092, -0.3127,  0.6586,
         -1.5467, -0.2833, -1.6969,  0.3793,  0.5257, -0.8488, -0.6607,  1.6476,
         -1.0540, -0.9293,  1.0357,  0.4577]], grad_fn=<SliceBackward>)

In [566]:
o1[:,0,:20]

tensor([[ 1.1431,  0.9443, -1.1135,  0.9790,  0.8699,  0.7742, -1.2484,  0.3398,
         -1.3116,  0.8435, -0.5928,  2.4487, -1.5294, -0.1209,  0.8277,  0.2153,
          0.4295, -0.5154, -0.3021, -1.0291],
        [ 1.9075, -0.5780, -0.1342,  0.4117,  1.6919, -0.7286, -1.1590,  0.3912,
         -0.8881, -0.0081,  0.2495,  0.3791,  0.6632, -0.2955, -0.8054, -0.4049,
          0.2013, -0.4917,  0.0273, -0.7945],
        [ 1.4742,  0.3332, -0.9849, -0.1476,  0.7846, -0.7092, -0.3127,  0.6586,
         -1.5467, -0.2833, -1.6969,  0.3793,  0.5257, -0.8488, -0.6607,  1.6476,
         -1.0540, -0.9293,  1.0357,  0.4577]], grad_fn=<SliceBackward>)